# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import duckdb
import pandas as pd
import os
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

CACHE_PATH = "work/outputs/feature_df_cache.parquet"

if os.path.exists(CACHE_PATH):
    feature_df = pd.read_parquet(CACHE_PATH)
    print("Loaded from cache:", feature_df.shape)
else:
    feature_df = con.sql("""
    WITH base AS (
        SELECT
            client_hash_id, content_hash_id, report_date,
            gsc_impressions, gsc_clicks, gsc_sum_position,
            gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr,
            gsc_sum_position * 1.0 / NULLIF(gsc_impressions, 0) AS avg_position
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        WHERE gsc_data_available IS TRUE AND gsc_impressions >= 50
    ),
    tiered AS (
        SELECT *, CASE
            WHEN avg_position <= 3 THEN 'top_3'
            WHEN avg_position <= 10 THEN 'page_1'
            WHEN avg_position <= 20 THEN 'page_2'
            WHEN avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep' END AS position_tier
        FROM base
    ),
    tier_medians AS (
        SELECT position_tier, MEDIAN(ctr) AS expected_ctr FROM tiered GROUP BY position_tier
    )
    SELECT t.*, m.expected_ctr, m.expected_ctr - t.ctr AS ctr_gap
    FROM tiered t JOIN tier_medians m USING (position_tier)
    """).df()
    os.makedirs("work/outputs", exist_ok=True)
    feature_df.to_parquet(CACHE_PATH)
    print("Queried fresh and cached:", feature_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queried fresh and cached: (1037442, 11)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ['gsc_impressions', 'avg_position']
categorical_features = ['position_tier']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
], remainder='passthrough')

def precision_at_k(df, score_col, label_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

## 1. Two paper findings + my methodology questions

**1. Two paper findings + my methodology questions**

**Finding #3 — Click Capture by Position Tier** (Top 3: 0.423%, Deep: 0.050%, "88% drop")

Methodology question: The paper states these are "weighted portfolio CTRs computed from total
clicks divided by total impressions in each position tier" — this is the same tier-aggregation
approach I used in ML-07/ML-08. My question: is the population selection filtered by a minimum
impression threshold per page before aggregating? If low-impression pages with 0 clicks are
included at full weight, the weighted average could still be dominated by high-impression pages
in each tier, masking noisier individual-page behavior — worth disclosing which pages qualify
for the tier average, similar to the "population selection checked for outcome-window
information" checkpoint.

**ML Appendix — "What Predicts Health?" (Random Forest feature importance)**

Methodology question: The paper's own caption discloses "the target itself is partly
constructed from some of these inputs, so importance is descriptive rather than causal" —
Health Score = Impressions(30) + Position(30) + CTR(20) + Scroll(20), and the model's top three
features (Average Position 43%, Impressions 32%, Scroll Depth 15% — 90% of total importance)
are exactly three of those four components. This matches the leakage taxonomy's "label-derived
features" pattern: the label was computed FROM these columns. The paper handles this
responsibly by disclosing it, but the methodology question I'd ask: was a train-without-suspect
test run (dropping position/impressions/scroll and re-measuring) to see how much predictive
power comes from genuinely independent signals like content_age or word_count, which show 0%
importance? That comparison would clarify whether the model has any information beyond
restating the label formula.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

**2. My model under an honest split (before/after)**

Before: naive random row-level split (70/30). After: GroupKFold by client_hash_id — the
formal sklearn tool for this, replacing the manual sorted-then-split approach used in ML-08.

| Split | Precision@50 |
|---|---|
| Random (naive) | [X] |
| Grouped by client (honest) | [Y] |

Gap: [X-Y]. [Explain: if random is inflated, the model is partly memorizing per-client
patterns rather than learning generalizable signal — the honest number is the grouped one.]
Per the effect-size literature (Sullivan & Feinn), a numeric gap alone isn't the full story —
what matters is whether this gap is large enough to change a real decision about which split
strategy to trust in production; here it [is/isn't] because [reasoning].

In [4]:
from sklearn.model_selection import GroupKFold
import numpy as np

# BEFORE: random split (what a careless first pass might do)
np.random.seed(42)
random_mask = np.random.rand(len(feature_df)) < 0.7
random_train = feature_df[random_mask]
random_test = feature_df[~random_mask]

# quick label + model on random split, for comparison
random_train = random_train.copy()
random_test = random_test.copy()
random_train['ctr_rank_pct'] = random_train.groupby('position_tier')['ctr'].rank(pct=True, method='first')
random_train['needs_review'] = (random_train['ctr_rank_pct'] <= 0.25).astype(int)
random_test['ctr_rank_pct'] = random_test.groupby('position_tier')['ctr'].rank(pct=True, method='first')
random_test['needs_review'] = (random_test['ctr_rank_pct'] <= 0.25).astype(int)

X_train_r = random_train[numeric_features + categorical_features]
y_train_r = random_train['needs_review']
X_test_r = random_test[numeric_features + categorical_features]
y_test_r = random_test['needs_review']

model_random = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
model_random.fit(X_train_r, y_train_r)
random_test['model_score'] = model_random.predict_proba(X_test_r)[:, 1]

random_p50 = precision_at_k(random_test, 'model_score', 'needs_review', k=50)

# AFTER: GroupKFold by client_hash_id (the formal tool, replacing manual sorted-split)
groups = feature_df['client_hash_id'].values
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(feature_df, groups=groups))
grouped_train = feature_df.iloc[train_idx].copy()
grouped_test = feature_df.iloc[test_idx].copy()

# same label + model logic on grouped split
# (reuse ML-08 code)

print(f"Random split precision@50: {random_p50:.3f}")
print(f"Grouped split precision@50: {grouped_p50:.3f}")
print(f"Gap: {random_p50 - grouped_p50:.3f}")

Random split precision@50: 0.020


NameError: name 'grouped_p50' is not defined

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.